# Classification de texte avec des RNN

Dans ces travaux pratiques, nous allons retrouver l'auteur d'une phrase avec des réseaux récurrents. Le corpus est celui des slides : les phrases de 2 œuvres de chacun de 9 auteurs anglophones, de 1821 à 1913 (Brontë, Melville, Wilde…).

1. Du texte aux séquences d'indices, avec `TextVectorization`.
2. Un classifieur récurrent qui ignore le *padding*.
3. L'entraînement d'un GRU bidirectionnel.
4. Sans masque, le GRU lit le *padding*.
5. GRU simple ou bidirectionnel ?
6. Des embeddings GloVe pré-entraînés et figés.
7. Un modèle plus profond.
8. Les auteurs que le modèle confond.

Ce TP a besoin d'une carte graphique : vérifiez dans `Exécution > Modifier le type d'exécution` que l'accélérateur matériel est un GPU. Chaque entraînement y prend de 10 secondes à une minute et quart (le modèle profond de la partie 7) sur une RTX 3090 ; le T4 de Colab est environ deux à trois fois plus lent. Sur CPU, il faudrait d'un quart d'heure à plusieurs heures.

## Téléchargements

Le corpus, le modèle de spaCy pour l'anglais et les vecteurs GloVe de la partie 6 (un fichier de 822 Mo) : comptez moins d'une minute.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh classical-authors
!python -m spacy download en_core_web_sm
!wget -q https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip
!unzip -q -o glove.6B.zip glove.6B.300d.txt

## Import des bibliothèques

In [ ]:
import pathlib
from collections.abc import Iterable, Iterator

import keras
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import spacy
import tqdm.auto
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Prétraitements

La cellule suivante prépare les données, comme dans les slides :

- chaque phrase, découpée par spaCy, est un exemple, et son étiquette est son auteur, de 0 à 8 ;
- toute entité nommée (un mot ou un groupe de mots) est remplacée par son type, première lettre triplée pour en faire un mot nouveau (`PPPERSON`, `DDDATE`…) : le modèle doit reconnaître un style, pas les noms des personnages ;
- les dossiers du corpus portent des noms à coquilles (`Melwille`, `bronteCharlote`) : `AUTEURS` donne le nom de chaque auteur ;
- une phrase sans aucun mot (un crochet « [ » isolé par spaCy, de la ponctuation seule) est écartée : vides une fois vectorisées, ces 1 763 phrases seraient entièrement masquées, ce que refuse cuDNN, le noyau des couches récurrentes de Keras sur GPU ;
- 70 % des phrases servent à l'entraînement (`X_train_raw`, `y_train`), 30 % au test (`X_test_raw`, `y_test`).

Elle prend environ une minute.

In [ ]:
# Dossier du corpus → nom de l'auteur
AUTEURS = {
  "bronteAnne": "Anne Brontë",
  "bronteCharlote": "Charlotte Brontë",
  "Cooper": "Cooper",
  "Emerson": "Emerson",
  "Hawthorne": "Hawthorne",
  "Melwille": "Melville",
  "Shaw": "Shaw",
  "Thoreau": "Thoreau",
  "wilde": "Wilde",
}

nlp = spacy.load(
  "en_core_web_sm",
  exclude=("tok2vec", "tagger", "parser", "attribute_ruler", "lemmatizer"),
)
nlp.enable_pipe("senter")


def remplace_entites(jetons: Iterable[spacy.tokens.Token]) -> Iterator[str]:
  for jeton in jetons:
    if jeton.ent_iob_ == "O":
      yield jeton.text
    elif jeton.ent_iob_ == "B":
      prefixe = jeton.ent_type_[0]
      yield f"{prefixe}{prefixe}{jeton.ent_type_}"


def lit_donnees(dossier: pathlib.Path) -> tuple[list[str], list[str]]:
  textes = []
  auteurs = []
  # <auteur>/<œuvre>/<chapitre>.txt
  chemins = sorted(dossier.glob("*/*/*.txt"))
  contenus = (p.read_text(encoding="utf8").replace("\n", " ") for p in chemins)
  for chemin, doc in tqdm.auto.tqdm(
    zip(chemins, nlp.pipe(contenus)), total=len(chemins)
  ):
    auteur = AUTEURS[chemin.parent.parent.name]
    for phrase in doc.sents:
      texte = " ".join(remplace_entites(phrase))
      # Une phrase sans lettre ni chiffre serait vide une fois vectorisée
      if any(caractere.isalnum() for caractere in texte):
        auteurs.append(auteur)
        textes.append(texte)
  return textes, auteurs


textes, auteurs = lit_donnees(pathlib.Path("classical-authors"))
encodeur_etiquettes = sklearn.preprocessing.LabelEncoder()
y = encodeur_etiquettes.fit_transform(auteurs)
print(
  f"{len(textes)} phrases ; étiquettes 0 à 8 : {', '.join(encodeur_etiquettes.classes_)}"
)

X_train_raw, X_test_raw, y_train, y_test = sklearn.model_selection.train_test_split(
  textes, y, test_size=0.3, random_state=0
)
print(f"Entraînement : {len(X_train_raw)} phrases, test : {len(X_test_raw)}")
for texte, etiquette in zip(X_train_raw[:3], y_train[:3]):
  print(f"{encodeur_etiquettes.classes_[etiquette]} : {texte}")

## 1. Du texte aux séquences d'indices

La cellule suivante transforme les phrases en tenseurs d'indices de mots, comme dans les slides, avec une couche `layers.TextVectorization`, `vectoriseur` : un vocabulaire des 20 000 mots les plus fréquents, appris sur les seules phrases d'entraînement et rangé dans `vocabulaire`, puis 150 indices par phrase, une phrase plus courte étant complétée par des 0 à la fin (le *padding*), une plus longue tronquée.

In [ ]:
vectoriseur = layers.TextVectorization(
  max_tokens=20_000,  # les 20 000 mots les plus fréquents
  output_sequence_length=150,  # 0 ajoutés à la fin, ou phrase tronquée
)
vectoriseur.adapt(X_train_raw)  # vocabulaire de l'entraînement seul
X_train = vectoriseur(X_train_raw)
X_test = vectoriseur(X_test_raw)
vocabulaire = vectoriseur.get_vocabulary()

print(X_train.shape)
print(X_train_raw[0])
print(X_train[0, :20].numpy())
print(" | ".join(vocabulaire[:10]))

### Combien de mots garder ?

Les deux choix de la cellule précédente, 150 indices par phrase et un vocabulaire de 20 000 mots, se décident d'après les données. Mesurez-les :

- `longueurs`, le nombre de mots de chaque phrase d'entraînement : les indices non nuls de sa ligne de `X_train` (une phrase de plus de 150 mots y compte pour 150) ;
- l'histogramme de `longueurs`, sa médiane et son 99e centile ;
- la part des phrases d'entraînement tronquées, celles qui remplissent leurs 150 indices ;
- la part des mots des phrases de test hors du vocabulaire (l'indice 1), et celle des phrases de test qui en ont au moins un.

Quelle longueur maximale choisiriez-vous d'après cette distribution ? Que perd-on avec 150 indices, et que coûtent-ils ? Les 20 000 mots du vocabulaire suffisent-ils ?

**Documentation utile** :

- https://numpy.org/doc/stable/reference/generated/numpy.count_nonzero.html
- https://numpy.org/doc/stable/reference/generated/numpy.percentile.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
# Votre code ici

#### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `np.count_nonzero(X_train, axis=1)` : le nombre d'indices non nuls de chaque ligne
- `plt.hist(longueurs, bins=75)` : 75 barres de 2 mots de large ; `plt.yscale("log")` rend visibles les phrases longues, rares
- `np.median(longueurs)`, `np.percentile(longueurs, 99)` : la moitié des phrases sont plus courtes que la médiane, 99 sur 100 que le 99e centile
- `(longueurs == 150).mean()` : la part des phrases tronquées
- `test = X_test.numpy()`, puis `(test == 1).sum() / (test != 0).sum()` : la part des mots inconnus ; `(test == 1).any(axis=1)` dit pour chaque phrase si elle en a au moins un

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
longueurs = np.count_nonzero(..., axis=1)
plt.hist(longueurs, bins=...)
plt.yscale("log")
plt.xlabel("Mots par phrase")
plt.ylabel("Phrases")
plt.show()
print(f"Médiane : {np.median(...):.0f} mots ; 99e centile : {np.percentile(...):.0f}")
print(f"Phrases tronquées : {(longueurs == ...).mean():.1%}")
test = X_test.numpy()
print(f"Mots inconnus en test : {(test == ...).sum() / (test != ...).sum():.1%}")
print(f"Phrases avec un mot inconnu : {(test == ...).any(axis=1).mean():.0%}")
```

#### Solution

In [ ]:
longueurs = np.count_nonzero(X_train, axis=1)  # mots de chaque phrase, 150 au plus
plt.hist(longueurs, bins=75)
plt.yscale("log")  # les phrases longues sont rares
plt.xlabel("Mots par phrase")
plt.ylabel("Phrases")
plt.show()
print(
  f"Médiane : {np.median(longueurs):.0f} mots ; "
  f"99e centile : {np.percentile(longueurs, 99):.0f}"
)
print(f"Phrases tronquées : {(longueurs == 150).mean():.1%}")
test = X_test.numpy()
print(f"Mots inconnus en test : {(test == 1).sum() / (test != 0).sum():.1%}")
print(f"Phrases avec un mot inconnu : {(test == 1).any(axis=1).mean():.0%}")

# Une phrase compte 17 mots en médiane, et 99 % en ont moins de 100 : le 99e centile,
# 100 indices, suffirait. 150 n'en tronquent que 0,2 % ; les 50 indices de plus ne
# coûtent pas d'exactitude, puisque le GRU saute le padding grâce au masque (partie 2),
# mais un peu de calcul : le GRU parcourt les 150 pas de chaque batch. Les 20 000 mots
# gardés, sur les 32 603 des phrases d'entraînement, couvrent 98 % des mots de test :
# 1,9 % sont inconnus (des mots rares, des noms que spaCy n'a pas reconnus). Mais 27 %
# des phrases de test en ont au moins un : un vocabulaire plus grand en sauverait
# quelques-uns, au prix d'une table d'embeddings plus grande, pour des mots trop rares
# pour y être bien appris.

## 2. Un classifieur récurrent qui ignore le *padding*

La fonction `construit_modele` de la cellule suivante est celle des slides, avec un paramètre de plus, `embedding` : la couche d'embedding, une nouvelle couche `layers.Embedding` de 64 dimensions si elle n'est pas donnée (les vecteurs GloVe de la partie 6). Les parties suivantes l'appellent avec la ou les couches récurrentes à comparer, par exemple `construit_modele(layers.GRU(64))`.

Le masque est indispensable ici : une phrase compte 17 mots en médiane, suivis de plus de 130 zéros de *padding*. `mask_zero=True` calcule le masque des indices 0, que l'embedding transmet aux couches récurrentes : elles sautent ces pas, et leur état final est celui du dernier vrai mot.

In [ ]:
def construit_modele(
  *couches_recurrentes: layers.Layer, embedding: layers.Embedding | None = None
) -> keras.Sequential:
  if embedding is None:
    # Un vecteur de 64 nombres par mot, et le masque des indices 0
    embedding = layers.Embedding(20_000, 64, mask_zero=True)
  return keras.Sequential(
    [
      keras.Input(shape=(None,), dtype="int32"),  # phrases de toute longueur
      embedding,
      *couches_recurrentes,
      layers.Dense(len(encodeur_etiquettes.classes_), activation="softmax"),
    ]
  )

## 3. Entraîner un GRU bidirectionnel

La fonction `entraine` compile un modèle, l'entraîne et l'évalue comme dans les slides :

- perte `sparse_categorical_crossentropy`, puisque les étiquettes sont des entiers ; optimiseur Adam, avec écrêtage de la norme du gradient (`clipnorm`) ;
- des batchs de 256 phrases ; 20 % des phrases d'entraînement en validation ;
- 20 epochs au plus : `EarlyStopping` arrête l'entraînement après 2 epochs sans baisse de la perte de validation, et remet les poids de la meilleure.

Elle affiche les courbes d'exactitude et renvoie l'exactitude de test. La cellule suivante l'applique à un GRU bidirectionnel de 64 unités par sens de lecture.

In [ ]:
def entraine(modele: keras.Model, nb_epoques: int = 20) -> float:
  modele.compile(
    optimizer=keras.optimizers.Adam(clipnorm=1.0),  # écrêtage du gradient
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  arret = keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)
  historique = modele.fit(
    X_train,
    y_train,
    batch_size=256,
    epochs=nb_epoques,
    validation_split=0.2,
    callbacks=[arret],
  )
  _, exactitude = modele.evaluate(X_test, y_test, verbose=0)

  # Exactitudes de la meilleure epoch, celle dont les poids sont gardés
  meilleur = int(np.argmin(historique.history["val_loss"]))
  print(
    f"Meilleure epoch : {meilleur + 1} ; exactitude d'entraînement "
    f"{historique.history['accuracy'][meilleur]:.3f}, "
    f"de validation {historique.history['val_accuracy'][meilleur]:.3f}, de test {exactitude:.3f}"
  )
  epoques_effectuees = np.arange(1, len(historique.epoch) + 1)
  plt.plot(epoques_effectuees, historique.history["accuracy"], label="entraînement")
  plt.plot(epoques_effectuees, historique.history["val_accuracy"], label="validation")
  plt.axvline(meilleur + 1, color="gray", linestyle="--", label="meilleure epoch")
  plt.xlabel("Epoch")
  plt.ylabel("Exactitude")
  plt.legend()
  plt.show()
  return exactitude

In [ ]:
bigru = construit_modele(layers.Bidirectional(layers.GRU(64)))
bigru.summary()
bigru_exactitude = entraine(bigru)

### Premières conclusions

Au vu des courbes et des exactitudes, que dire de la qualité de cet apprentissage ? Comparez l'exactitude de test au hasard (une chance sur 9) et à un modèle qui prédirait toujours l'auteur le plus fréquent, Melville (18 % des phrases).

Votre réponse ici

#### Solution

Le modèle apprend : environ 55 % d'exactitude en test, cinq fois le hasard (11 %) et trois fois la classe majoritaire (18 %).

Mais il surapprend dès la 3e epoch : son exactitude d'entraînement continue de monter (75 % à la 4e epoch), alors que celle de validation plafonne vers 57 % et que la perte de validation remonte. `EarlyStopping` arrête l'entraînement deux epochs plus tard et remet les poids de la meilleure epoch.

Le modèle a 1,3 million de paramètres, dont 1,28 million dans la table d'embeddings, pour 41 000 phrases d'entraînement : il retient les mots des phrases d'entraînement plutôt que le style de chaque auteur.

## 4. Sans masque, le GRU lit le *padding*

Que gagne-t-on au masque ? Entraînez avec `entraine` le même GRU bidirectionnel que dans la partie 3, mais sur une couche `layers.Embedding` sans masque : l'indice 0 y devient un mot comme un autre, que le GRU lit. Comparez leurs exactitudes de test.

Dans `X_train`, une phrase de 17 mots est suivie de 133 zéros. Par quoi chaque sens de lecture du GRU finit-il sa lecture ? Lequel des deux le *padding* gêne-t-il ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `layers.Embedding(20_000, 64)` : sans `mask_zero=True`, aucun masque n'est calculé, ni transmis au GRU
- `construit_modele` prend la couche d'embedding par son paramètre `embedding`, après les couches récurrentes
- `bigru_exactitude` contient l'exactitude de test du GRU bidirectionnel de la partie 3

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bigru_sans_masque = construit_modele(
  layers.Bidirectional(...), embedding=layers.Embedding(..., ...)
)
exactitude_sans_masque = entraine(...)
print(f"Avec masque : {bigru_exactitude:.3f}, padding lu : {...:.3f}")
```

### Solution

In [ ]:
bigru_sans_masque = construit_modele(
  layers.Bidirectional(layers.GRU(64)),
  embedding=layers.Embedding(20_000, 64),  # sans mask_zero : le padding est lu
)
exactitude_sans_masque = entraine(bigru_sans_masque)
print(
  f"Avec masque : {bigru_exactitude:.3f}, padding lu : {exactitude_sans_masque:.3f}"
)

Sans masque, le GRU bidirectionnel fait un peu moins bien : environ 53 % d'exactitude en test, contre 55 % avec le masque.

Son sens avant lit la phrase, puis ses 133 zéros : son état final est celui du dernier zéro, et ce qu'il a retenu de la phrase doit traverser plus de 130 pas sans information. Son sens arrière, lui, commence par les zéros, puis lit la phrase à l'envers : son état final est celui du premier mot, et il résume encore toute la phrase. Le *padding* gêne donc surtout le sens avant, et le modèle ne s'appuie plus guère que sur le sens arrière : son exactitude est à peu près celle du GRU d'un seul sens de lecture de la partie 5. L'écart reste modeste, parce que le sens arrière suffit presque ; un GRU simple, lui, n'aurait que son sens avant, et le masque y compterait bien plus.

## 5. GRU simple ou bidirectionnel ?

Entraînez avec `entraine` un GRU simple de 64 unités, qui ne lit les phrases que dans un sens, et comparez-le au GRU bidirectionnel de la partie 3 :

- l'exactitude de test ;
- le nombre de paramètres de la couche récurrente, et du modèle entier ;
- la durée d'une epoch, que Keras affiche pendant l'entraînement.

Le second sens de lecture vaut-il son coût ici ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

- `construit_modele(layers.GRU(64))` : le même modèle, avec un GRU simple
- `modele.summary()` donne le nombre de paramètres de chaque couche ; `modele.count_params()` celui du modèle entier
- `bigru_exactitude` contient l'exactitude de test du GRU bidirectionnel

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
gru = construit_modele(...)
gru.summary()
gru_exactitude = entraine(...)
print(f"GRU : {gru_exactitude:.3f}, {gru.count_params()} paramètres")
print(f"GRU bidirectionnel : {...:.3f}, {bigru.count_params()} paramètres")
```

### Solution

In [ ]:
gru = construit_modele(layers.GRU(64))
gru.summary()
gru_exactitude = entraine(gru)
print(f"GRU : {gru_exactitude:.3f}, {gru.count_params()} paramètres")
print(f"GRU bidirectionnel : {bigru_exactitude:.3f}, {bigru.count_params()} paramètres")

Le GRU simple fait un peu moins bien : environ 53 % d'exactitude en test, contre 54 à 56 % pour le GRU bidirectionnel selon les exécutions.

Le second sens de lecture double les paramètres de la couche récurrente (24 960 → 49 920), mais ce n'est que 2 % du modèle, dont l'embedding fait l'essentiel (1,28 million) ; une epoch dure environ 40 % de plus (11 ms par batch contre 8 sur une RTX 3090). Le gain est modeste, mais il vaut ce coût : l'état final d'un GRU simple résume surtout la fin de la phrase, et la direction arrière y ajoute un résumé de son début.

## 6. Des embeddings GloVe pré-entraînés et figés

Les embeddings des modèles précédents partent de vecteurs aléatoires et sont appris sur nos 41 000 phrases d'entraînement. Les vecteurs [GloVe](https://nlp.stanford.edu/projects/glove/), eux, ont été appris sur 6 milliards de mots (Wikipédia et Gigaword) : des mots de sens proche y ont des vecteurs proches.

Lisez le fichier `glove.6B.300d.txt` dans un dictionnaire `glove`, mot → vecteur : chaque ligne est un mot suivi de ses 300 coordonnées, séparés par des espaces, et seuls les mots de `vocabulaire` nous intéressent. La cellule qui suit la solution en fait une couche d'embedding, comme dans la slide « Embeddings pré-entraînés : GloVe ».

Combien des 20 000 mots du vocabulaire GloVe connaît-il ? Lesquels manquent, et pourquoi ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `mot, *coordonnees = ligne.rstrip().split(" ")` sépare le mot de ses coordonnées ; `np.array(coordonnees, dtype="float32")` en fait un vecteur
- `set(vocabulaire)` : un test `mot in ...` rapide sur les 400 000 lignes du fichier

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
mots = set(vocabulaire)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as fichier:
  for ligne in fichier:
    mot, *coordonnees = ligne.rstrip().split(" ")
    if mot in ...:
      glove[mot] = np.array(coordonnees, dtype="float32")

absents = [mot for mot in vocabulaire if mot not in ...]
print(f"{len(glove)} mots sur {len(vocabulaire)} dans GloVe")
print("Absents :", ", ".join(absents[:30]))
```

### Solution

In [ ]:
# Seuls les vecteurs des mots du vocabulaire sont gardés
mots = set(vocabulaire)
glove = {}
with open("glove.6B.300d.txt", encoding="utf8") as fichier:
  for ligne in fichier:
    mot, *coordonnees = ligne.rstrip().split(" ")
    if mot in mots:
      glove[mot] = np.array(coordonnees, dtype="float32")

absents = [mot for mot in vocabulaire if mot not in glove]
print(f"{len(glove)} mots sur {len(vocabulaire)} dans GloVe")
print("Absents :", ", ".join(absents[:30]))

# 19 102 des 20 000 mots du vocabulaire sont dans GloVe. Absents : le padding et [UNK]
# (indices 0 et 1), les types d'entités (ppperson, dddate…), des noms que spaCy n'a pas
# reconnus (holgrave, pyncheon), le vocabulaire de la chasse à la baleine (harpooneer,
# whalemen) et des graphies anciennes (oftener, trowsers, noontide) : des mots qu'un
# corpus du XXIe siècle (Wikipédia, la presse) n'emploie pas.

### La couche d'embedding GloVe

La cellule suivante construit `embedding_glove` comme dans la slide « Embeddings pré-entraînés : GloVe » :

- `matrice_embedding`, de forme `(len(vocabulaire), 300)` : sa ligne `i` est le vecteur du mot `vocabulaire[i]`, ou reste nulle si GloVe ne connaît pas ce mot ;
- une couche `layers.Embedding` initialisée avec cette matrice (`embeddings_initializer`), figée (`trainable=False` : ses vecteurs ne changent pas pendant l'entraînement), et qui calcule le masque du *padding* (`mask_zero=True`).

In [ ]:
matrice_embedding = np.zeros((len(vocabulaire), 300))  # 0 si le mot est absent
for i, mot in enumerate(vocabulaire):
  if mot in glove:
    matrice_embedding[i] = glove[mot]
embedding_glove = layers.Embedding(
  input_dim=len(vocabulaire),
  output_dim=300,
  mask_zero=True,
  embeddings_initializer=keras.initializers.Constant(matrice_embedding),
  trainable=False,  # table figée
)

### Entraîner avec GloVe

Entraînez le même GRU bidirectionnel que dans la partie 3, avec `embedding_glove` pour couche d'embedding, et comparez-le au premier :

- l'écart entre l'exactitude d'entraînement et celle de validation ;
- l'exactitude de test ;
- le nombre de paramètres entraînables, que `summary()` donne.

In [ ]:
# Votre code ici

#### Conseils supplémentaires

`construit_modele` prend la couche d'embedding par son paramètre `embedding`, après les couches récurrentes. `bigru_exactitude` contient l'exactitude de test du premier modèle.

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
bigru_glove = construit_modele(layers.Bidirectional(...), embedding=...)
bigru_glove.summary()
glove_exactitude = entraine(...)
print(
  f"Embeddings appris : {bigru_exactitude:.3f}, GloVe figé : {glove_exactitude:.3f}"
)
```

#### Solution

In [ ]:
bigru_glove = construit_modele(
  layers.Bidirectional(layers.GRU(64)), embedding=embedding_glove
)
bigru_glove.summary()
glove_exactitude = entraine(bigru_glove)
print(
  f"Embeddings appris : {bigru_exactitude:.3f}, GloVe figé : {glove_exactitude:.3f}"
)

Avec GloVe figé, le modèle n'a plus que 141 705 paramètres entraînables, contre 1,3 million : les 6 millions de nombres de la table GloVe ne bougent pas.

Il surapprend bien plus lentement : sa perte de validation baisse pendant 8 epochs au lieu de 2, jusqu'à un minimum plus bas, et l'exactitude de test progresse un peu, d'environ 55 % à 57 %. Les vecteurs GloVe apportent ce que nos 41 000 phrases ne peuvent pas apprendre seules : des mots de sens proche y ont des vecteurs proches, même quand ils sont rares dans le corpus.

L'écart entre l'exactitude d'entraînement et celle de validation finit pourtant par se creuser : la couche récurrente, elle, apprend toujours par cœur. C'est le rôle du *dropout* du modèle suivant.

## 7. Un modèle plus profond

Un dernier modèle, donné : deux GRU bidirectionnels empilés sur les embeddings GloVe.

- `return_sequences=True` : la première couche renvoie un état par mot, la séquence que lit la seconde ;
- `dropout=0.5` : la moitié des valeurs d'entrée de chaque couche récurrente mises à zéro à chaque pas d'entraînement, dans les deux sens de lecture, contre le surapprentissage.

In [ ]:
profond = construit_modele(
  layers.Bidirectional(layers.GRU(64, dropout=0.5, return_sequences=True)),
  layers.Bidirectional(layers.GRU(64, dropout=0.5)),
  embedding=embedding_glove,
)
profond.summary()
profond_exactitude = entraine(profond)
print(
  f"GloVe et un GRU bidirectionnel : {glove_exactitude:.3f}, deux : {profond_exactitude:.3f}"
)

Le modèle profond atteint environ 58 % d'exactitude en test, à peine plus que le GRU bidirectionnel sur GloVe. Le *dropout* a supprimé le surapprentissage : les exactitudes d'entraînement et de validation restent égales, vers 59 % à la fin. Mais il ralentit l'apprentissage : la perte de validation baisse encore à la 20e epoch, et `EarlyStopping` n'a pas eu à arrêter l'entraînement. Quelques epochs de plus gagneraient sans doute encore un peu.

## 8. Les auteurs que le modèle confond

La cellule suivante affiche la matrice de confusion du modèle profond sur les phrases de test : la ligne d'un auteur donne la répartition de ses phrases entre les auteurs prédits.

In [ ]:
y_pred = profond.predict(X_test, batch_size=1024).argmax(axis=1)
fig, ax = plt.subplots(figsize=(8, 8))
sklearn.metrics.ConfusionMatrixDisplay.from_predictions(
  y_test,
  y_pred,
  display_labels=encodeur_etiquettes.classes_,
  normalize="true",  # proportions de chaque ligne
  values_format=".0%",
  xticks_rotation=45,
  cmap="Blues",
  colorbar=False,
  ax=ax,
)
ax.set_xlabel("Auteur prédit")
ax.set_ylabel("Auteur réel")
plt.show()

### Lecture de la matrice

Quels auteurs le modèle reconnaît-il le mieux ? Lesquels confond-il, et qu'ont-ils en commun ? Les œuvres du corpus sont listées dans le [README du jeu de données](https://github.com/shuuchuu/datasets/tree/main/classical-authors).

Votre réponse ici

#### Solution

La diagonale donne la part des phrases de chaque auteur bien attribuées ; ces chiffres varient de quelques points d'une exécution à l'autre.

- Wilde (68 %), Melville (65 %) et Thoreau (64 %) sont les mieux reconnus. *Moby-Dick* a un vocabulaire à part (la mer, la chasse à la baleine), les deux livres de Thoreau aussi (la nature, la rivière, les bois), et les pièces de Wilde sont faites de répliques courtes, au style oral, qui ne ressemblent ni aux romans ni aux essais.
- Anne Brontë est la moins bien reconnue (43 %) : 24 % de ses phrases sont attribuées à sa sœur Charlotte. Ce sont deux romancières anglaises de la même époque, aux romans racontés à la première personne, par une gouvernante dans *Agnes Grey* comme dans *Jane Eyre*. La confusion va surtout dans ce sens (7 % des phrases de Charlotte seulement vont à Anne) : Charlotte, presque deux fois plus représentée, attire les phrases ambiguës, et pas seulement celles d'Anne (de 4 à 10 % des phrases de chacun des autres auteurs).
- Shaw et Wilde se confondent : 18 % des phrases de Shaw sont prises pour du Wilde, 13 % de celles de Wilde pour du Shaw. Ce sont les deux dramaturges irlandais du corpus, et leurs répliques se ressemblent.
- Emerson et Thoreau aussi : 22 % des phrases d'Emerson sont attribuées à Thoreau, 12 % dans l'autre sens. Ce sont deux essayistes américains du transcendantalisme, voisins et amis.

Une phrase isolée est un indice faible : beaucoup sont courtes et banales (« Yes . »), et aucun modèle ne peut les attribuer. Pour situer ces résultats, une régression logistique sur une représentation TF-IDF des phrases atteint environ 61 % d'exactitude sur ce même découpage : sur des phrases courtes, les mots employés comptent plus que leur ordre.